# Computing Word-Level Reading Measures from Eye-Tracking Data

This tutorial demonstrates how to compute AOI-based word-level reading measures from eye-tracking data using predefined Areas of Interest (AOIs).

## What you will learn

In this tutorial, you will learn how to:

- load a DataFrame containing fixation events,
- load a DataFrame defining word-level AOIs with their bounding boxes,
- map fixations to the corresponding AOIs,
- compute word-level reading measures using `compute_reading_measures`,
- inspect the resulting DataFrame of computed reading measures, and
- compute reading measures for several stimuli at once using `group_columns` and a dict of AOI definitions.

In [ ]:
import polars as pl

from pymovements import Dataset, Events
from pymovements.measure.reading import compute_reading_measures
from pymovements.stimulus import TextStimulus

We begin by loading a dataset containing fixation events together with the corresponding AOI definitions. In this tutorial, we use the `GGTG` dataset, which can be loaded as follows:

In [ ]:
dataset = Dataset('GGTG', path='data/GGTG')

# Download the dataset and extract all archives.
dataset.download()

# Load the dataset into memory for processing
dataset.load(subset={'subject_id': 'P01'})

For simplicity, we restrict the analysis to a single subject and a single stimulus. Specifically, we use the first subject and the stimulus `goldfish-pos.text.0`. We then select only the required columns and add a column indicating the event type:

In [ ]:
stimulus = "goldfish-pos.text.0"
sample_fixation_path = dataset.paths.precomputed_events / \
    dataset.fileinfo['precomputed_events']['filepath'][0]

fixations = pl.read_csv(sample_fixation_path)
fixations = fixations.filter(pl.col('stimulus') == stimulus)
fixations = fixations[['onset',
                       'offset',
                       'duration',
                       'location_x',
                       'location_y',
                       ]]
# add name column for the type of event
fixations = fixations.with_columns(name=pl.lit('fixation'))
fixations.head()

Next, we load the CSV file containing the AOI definitions for the selected stimulus:

In [ ]:
stimulus_rel_path = dataset.fileinfo['textstimulus'].filter(
    pl.col('stimulus') == stimulus).filter(
        pl.col('unit') == 'word')['filepath'][0]
aoi_path = dataset.paths.stimuli / stimulus_rel_path

aoi_df = pl.read_csv(aoi_path, separator=',')
aoi_df = aoi_df.with_columns(aoi_index=pl.col('index'))
aoi_df.head()

To use the AOI definitions, we convert the DataFrame into a `TextStimulus`:

In [ ]:
aoi_text_stimulus = TextStimulus(
    aoi_df,
    aoi_column='content',
    start_x_column='left',
    start_y_column='top',
    end_x_column='right',
    end_y_column='bottom',)
aoi_text_stimulus

Next, we map the fixation events to their corresponding AOIs. To do so, we create an `Events` DataFrame and use the `map_to_aois` function:

In [ ]:
events = Events(data=fixations)
events.map_to_aois(aoi_text_stimulus)
events.frame

With all required data prepared, we can now compute the reading measures using `compute_reading_measures`:

In [ ]:
rm_df = compute_reading_measures(
    fixations=events.frame,
    aois=aoi_df,
    word_index_column='aoi_index',
    word_column='content'
)
rm_df.head()

The resulting DataFrame contains one row per word (Area of Interest, AOI), along with a range of eye-tracking reading measures. A complete description of every output column can be found in the API documentation of {py:func}`~pymovements.measure.reading.compute_reading_measures`.

## Computing reading measures for several stimuli at once

So far we have processed a single reading sequence. When fixations from several trials or stimuli are combined in one DataFrame, `compute_reading_measures` keeps the independent reading sequences apart via its `group_columns` parameter. Each group needs its own AOI definitions, which are passed as a dict mapping the values of the first group column to the corresponding AOI DataFrame.

We prepare fixations for two stimuli, this time keeping the `stimulus` column, and map each stimulus to its own AOIs:

In [ ]:
stimuli = ['goldfish-pos.text.0', 'goldfish-pos.text.1']

multi_fixations = []
aois = {}
for stim in stimuli:
    stim_fixations = (
        pl.read_csv(sample_fixation_path)
        .filter(pl.col('stimulus') == stim)
        .select(['stimulus', 'onset', 'offset', 'duration', 'location_x', 'location_y'])
        .with_columns(name=pl.lit('fixation'))
    )

    stim_rel_path = dataset.fileinfo['textstimulus'].filter(
        pl.col('stimulus') == stim).filter(
            pl.col('unit') == 'word')['filepath'][0]
    stim_aois = pl.read_csv(dataset.paths.stimuli / stim_rel_path, separator=',')
    stim_aois = stim_aois.with_columns(aoi_index=pl.col('index'))
    aois[stim] = stim_aois

    stim_events = Events(data=stim_fixations)
    stim_events.map_to_aois(TextStimulus(
        stim_aois,
        aoi_column='content',
        start_x_column='left',
        start_y_column='top',
        end_x_column='right',
        end_y_column='bottom',))
    multi_fixations.append(stim_events.frame)

multi_fixations = pl.concat(multi_fixations)

In [ ]:
multi_rm_df = compute_reading_measures(
    fixations=multi_fixations,
    aois=aois,
    word_index_column='aoi_index',
    word_column='content',
    group_columns=['stimulus'],
)
multi_rm_df

The resulting DataFrame contains one row per stimulus and word, with the `stimulus` column preserved in the output.

### What you have learned in this tutorial:


- load fixation events into a DataFrame,
- load word-level AOI definitions and their bounding boxes,
- map fixation events to the corresponding AOIs,
- compute word-level reading measures using `compute_reading_measures`,
- inspect the resulting DataFrame containing the computed reading measures, and
- compute reading measures for several stimuli at once using `group_columns` and a dict of AOI definitions.